## 5.0 Contexto del notebook

Este notebook continúa lo aprendido en el Notebook 4:  
**las métricas describen comportamiento, pero no definen decisiones.**

Aquí damos el salto operativo:

> convertir **señales** (probabilidades / scores) en **políticas explícitas** usando thresholds.

### Artefactos que entran
Trabajaremos con outputs ya generados (sin reentrenar):

- `y_test` (verdad terreno)
- `y_proba` (probabilidades por clase) y/o `scores` (señal cruda)
- *(opcional)* `conf_df` si quieres reutilizar el análisis fila a fila

### Objetivo del notebook
- Definir un **evento operativo** (ej.: “vino bueno” vs “no bueno”).
- Construir una **regla de decisión** basada en threshold.
- Medir el trade-off real entre errores con **precision** y **recall**.
- Demostrar que:

📌 **El threshold es una decisión de negocio, no un ajuste técnico.**

---
---
---

## 5.1 Pasar de multiclase a “evento operativo”

Hasta ahora el modelo predice **calidades de vino** (3 a 9).  
Eso es útil para análisis, pero **no es directamente accionable**.

En la práctica, una organización no decide sobre “todas las clases”,  
decide sobre **eventos concretos**.

Ejemplos reales:
- ¿Este vino es **lo suficientemente bueno** para aprobarlo?
- ¿Este vino es **riesgoso** y requiere revisión?
- ¿Este vino entra o no en una categoría comercial?

Por eso, el primer paso operativo es **reformular el problema**.

<br>

### De clasificación multiclase a evento binario

Aquí tomamos una decisión explícita y documentada:

> **Evento operativo:**  
> Un vino se considera **“bueno”** si su calidad es **mayor o igual a 7**.

Esto implica:
- Clases **7, 8 y 9** → evento positivo (`1`)
- Clases **3, 4, 5 y 6** → evento negativo (`0`)

No estamos diciendo que esta sea “la mejor” definición.  
Estamos diciendo que **es una definición clara**, necesaria para avanzar.

📌 Sin evento operativo definido, **no existen thresholds**.  
📌 Sin threshold, **no existe política**.

<br>

### Qué cambia a partir de aquí

A partir de este punto:
- dejamos de evaluar “qué clase predice el modelo”,
- empezamos a evaluar **si detecta correctamente el evento**.

Esto nos permitirá:
- hablar de **precision y recall**,
- mover thresholds conscientemente,
- y discutir errores en términos de **impacto real**, no solo de métricas.

En el siguiente bloque construiremos la variable binaria (`y_true_bin`)  
que representa este evento operativo.

---
---
---

### Carga de artefactos del modelo

Este notebook **no entrena modelos**.  
Carga los artefactos generados en el Notebook 3 para analizarlos desde el punto de vista de métricas.

Los objetos cargados representan:
- resultados finales del modelo,
- señales ya producidas,
- datos inmutables para el análisis.

In [2]:
import numpy as np

# Carga de artefactos persistidos en el Notebook 3
y_test = np.load("../artifacts/y_test.npy")
y_pred = np.load("../artifacts/y_pred.npy")
y_proba = np.load("../artifacts/y_proba.npy")
scores = np.load("../artifacts/scores.npy")

y_test.shape, y_pred.shape, y_proba.shape, scores.shape

((975,), (975,), (975, 7), (975, 7))

---
---
---

## 5.1 Construcción del evento operativo

En este bloque se traduce la **variable multiclase** (`y_test`) a un **evento binario accionable**.

Definimos explícitamente qué significa un resultado **positivo** desde el punto de vista operativo:
- `1` → vino considerado **bueno** (calidad ≥ 7)
- `0` → vino **no bueno** (calidad < 7)

Este paso es clave porque:
- las métricas como *precision* y *recall* **solo existen** una vez definido el evento,
- los thresholds **no se aplican a clases**, se aplican a eventos,
- toda política de decisión parte de esta definición.

📌 Aquí no se optimiza nada.  
📌 Solo se fija, de forma explícita, **qué problema estamos decidiendo**.

In [3]:
# Construcción del evento operativo (vino "bueno")
# 1 = calidad >= 7, 0 = resto
y_true_bin = (y_test >= 7).astype(int)

# Verificación rápida
y_true_bin[:10]

array([0, 0, 0, 0, 1, 1, 0, 0, 0, 0])